# BTP · FBBench zero-shot runKaggle notebook that clones the repo from GitHub, attaches the cached frames,runs the benchmark on a GPU, and pushes results back to GitHub.**Before running, in the right-hand Settings panel:**| Setting | Value ||---|---|| Accelerator | GPU P100 (or T4 x2) || Internet | **On** — required for git and model download || Persistence | Variables and files |**And under Add-ons -> Secrets, add:**| Secret | Value ||---|---|| `GH_TOKEN` | GitHub fine-grained PAT, Contents: read & write, scoped to this repo only || `HF_TOKEN` | HuggingFace token (only if a gated model is used) |Keep this notebook **private**. Secrets are not printed unless you print them --never `print(TOKEN)`, and never paste a token into a cell.

## 1 · Config — edit these three lines

In [ ]:
GH_USER = "your-github-username"GH_REPO = "badminton-agentic-vlm"BRANCH  = "main"# Attached Kaggle Dataset holding the pre-extracted frames.# Leave as None on the first run, before the cache dataset exists.FRAME_DS = "/kaggle/input/finebadminton-frames"EXPERIMENT = "e01_zeroshot"MODEL_ID   = "Qwen/Qwen2.5-VL-7B-Instruct"N_QUESTIONS = 500

## 2 · Secrets and repo clone

In [ ]:
import os, subprocess, pathlib, datetimefrom kaggle_secrets import UserSecretsClientsec = UserSecretsClient()GH_TOKEN = sec.get_secret("GH_TOKEN")WORK = pathlib.Path("/kaggle/working")REPO = WORK / GH_REPOdef sh(cmd, cwd=None, check=True):    r = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str),                       capture_output=True, text=True)    # scrub the token so it can never reach the notebook output    out = (r.stdout + r.stderr).replace(GH_TOKEN, "***")    print(out[-3000:])    if check and r.returncode != 0:        raise RuntimeError(f"failed: {cmd}")    return rAUTH_URL = f"https://{GH_TOKEN}@github.com/{GH_USER}/{GH_REPO}.git"if REPO.exists():    sh(["git", "fetch", "origin"], cwd=REPO)    sh(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO)else:    sh(["git", "clone", "--depth", "1", "-b", BRANCH, AUTH_URL, str(REPO)])sh(["git", "config", "user.email", "btp@iiitkottayam.ac.in"], cwd=REPO)sh(["git", "config", "user.name",  "BTP Kaggle Runner"], cwd=REPO)print("repo at:", REPO)

## 3 · Environment

In [ ]:
!pip install -q --upgrade "transformers>=4.51" accelerate qwen-vl-utils decord 2>&1 | tail -2import torchprint("cuda:", torch.cuda.is_available())if torch.cuda.is_available():    p = torch.cuda.get_device_properties(0)    print(f"{p.name}  {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}")    # P100 = sm_60, T4 = sm_75, A100 = sm_80. bf16 needs sm_80+.    DTYPE = torch.bfloat16 if p.major >= 8 else torch.float16else:    DTYPE = torch.float32print("dtype:", DTYPE)

## 4 · DataFrames come from an attached Kaggle Dataset, not a download. Extraction is CPUwork -- do it once on your laptop, upload the result, and every run after thatattaches it in seconds instead of spending GPU quota on video decoding.

In [ ]:
import pathlibif FRAME_DS and pathlib.Path(FRAME_DS).exists():    FRAMES = pathlib.Path(FRAME_DS)    n = sum(1 for _ in FRAMES.rglob("*.jpg"))    print(f"attached frame cache: {FRAMES}  ({n} frames)")else:    # first run only -- pull the annotations so the cache can be built    print("no frame cache attached; downloading annotations")    !pip install -q "huggingface_hub[cli]"    !hf download Moujuruo/FineBadmintonBenchmark --repo-type dataset \         --local-dir /kaggle/working/fbbench 2>&1 | tail -2    FRAMES = None

## 5 · Sanity check — mock mode, no GPU, no excuses

In [ ]:
import syssys.path.insert(0, str(REPO / "src"))from bavlm.models.backend import build_backend, extract_choicefrom bavlm.eval.fbbench import load_questions   # phase 1questions = load_questions(limit=200)key = {q.qid: q.answer for q in questions}for mode, expect in [("oracle", "100%"), ("random", "~25%")]:    be = build_backend({"mock": mode, "seed": 0}, answer_key=key)    acc = sum(extract_choice(be.generate([], q.prompt, qid=q.qid)) == q.answer              for q in questions) / len(questions)    print(f"{mode:8} {acc:6.1%}   expected {expect}")# STOP if oracle is not 100% and random is not near chance.# Anything else means the scoring plumbing is broken and the real run would# produce a number that looks fine and is wrong.

## 6 · The run — resumable

In [ ]:
import json, time, datetimeRESULTS = REPO / "results" / f"{EXPERIMENT}_{MODEL_ID.split('/')[-1]}.jsonl"RESULTS.parent.mkdir(parents=True, exist_ok=True)# resume: skip anything already answered, so a 12-hour timeout costs nothingdone = set()if RESULTS.exists():    with open(RESULTS) as fh:        done = {json.loads(l)["qid"] for l in fh if l.strip()}    print(f"resuming, {len(done)} already done")backend = build_backend({"hf_id": MODEL_ID, "dtype": str(DTYPE).split('.')[-1]})todo = [q for q in load_questions(limit=N_QUESTIONS) if q.qid not in done]# group by clip so vLLM / HF prefix caching hits on the shared visual prefixtodo.sort(key=lambda q: q.clip_id)t0 = time.time()with open(RESULTS, "a") as fh:    for i, q in enumerate(todo, 1):        t = time.time()        frames = q.load_frames(FRAMES)        out = backend.generate(frames, q.prompt)        fh.write(json.dumps({            "qid": q.qid, "clip_id": q.clip_id, "domain": q.domain,            "gold": q.answer, "raw": out, "pred": extract_choice(out),            "seconds": round(time.time() - t, 3),        }) + "\n")        fh.flush()                      # survive a kill at any moment        if i % 25 == 0:            print(f"{i}/{len(todo)}  {(time.time()-t0)/60:.1f} min elapsed")print("done in", round((time.time() - t0) / 60, 1), "min")

## 7 · Score, per domain

In [ ]:
import pandas as pddf = pd.read_json(RESULTS, lines=True)df["correct"] = df["pred"] == df["gold"]by_domain = (df.groupby("domain")               .agg(n=("correct", "size"), accuracy=("correct", "mean"),                    sec_per_q=("seconds", "mean"))               .round(3))print(by_domain.to_string())print(f"\nOVERALL  {df['correct'].mean():.1%}  (n={len(df)})")print(f"unparsed {df['pred'].isna().sum()} responses")# Reference points from FineBadminton (ACM MM 2025):#   7B zero-shot      < 25%#   Gemini 2.5 Pro + S  38.6%#   Qwen2.5-VL-7B ft    42.06%

## 8 · Push results to GitHub

In [ ]:
stamp = datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")commit_sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"],                            cwd=REPO, capture_output=True, text=True).stdout.strip()# every reported number gets a manifest row -- no row, no reportrow = (f"| {stamp} | {EXPERIMENT} | {MODEL_ID} | uniform 2 FPS | "       f"{len(df)} | FBBench | {commit_sha} | {RESULTS.name} | "       f"{df['correct'].mean():.3f} |\n")with open(REPO / "results" / "MANIFEST.md", "a") as fh:    fh.write(row)by_domain.to_csv(REPO / "results" / f"{EXPERIMENT}_by_domain.csv")sh(["git", "add", "results/"], cwd=REPO)sh(["git", "commit", "-m",    f"results: {EXPERIMENT} {MODEL_ID.split('/')[-1]} n={len(df)} "    f"acc={df['correct'].mean():.3f}"], cwd=REPO, check=False)sh(["git", "push", "origin", BRANCH], cwd=REPO)print("pushed")

## 9 · Before you close the tabClick **Save Version** (top right). That commits `/kaggle/working` so the cloneand any un-pushed output survive. Results already on GitHub are safe either way-- which is the point of pushing them in step 8 rather than relying on Kaggle.